# RAG search, from the big picture to the parts

1. What RAG is, and where search fits
2. How we score search
3. The search techniques
4. Each one hands-on on our data: BM25 → dense embeddings → page-image search → hybrid → reranking
5. Side-by-side comparison

## 1. What RAG is

A language model only knows what it saw in training. **RAG (retrieval-augmented generation)** hands it the
relevant documents at question time: **retrieve** the pages that answer the question, then **generate** the
answer from them.

Why not just give the model everything?
- Your documents were never in its training data.
- It cannot read them all at once: 2,313 pages ≈ 3 million tokens; it holds about 16 thousand.
- With retrieved pages, the answer can **cite its source**, so you can check it.

![RAG: build an index once; per question retrieve, rerank, generate](figures/rag_overview.png)

- **Once:** turn every page into an index (word counts, vectors) so it can be searched fast.
- **Per question:** retrieve about 20 candidates fast → rerank them carefully, keep the best 3 → the model reads them and answers.

**Search caps everything:** if the right page is not retrieved, the model cannot use it.
Our project is **visual RAG**: the model reads pages as images, so charts and tables count.

**Learn more:** [Pinecone: Retrieval-Augmented Generation](https://www.pinecone.io/learn/retrieval-augmented-generation/) · deeper: [the original RAG paper](https://arxiv.org/abs/2005.11401)

## 2. How we score search

Each question's right pages are graded **2** (full answer) or **1** (part of it). Example:

![Ten ranked pages with the right ones at ranks 2, 4 and 7, and the four scores](figures/metrics_v2.png)

| metric | question it answers | weak spot |
|---|---|---|
| **Hit@5** | Is any right page in the top 5? | Ignores how many and how high |
| **Recall@5** | What share of the right pages is in the top 5? | Can't reach 1 when a question has more than 5 right pages (29 of 108 do) |
| **MRR@10** | How high is the first right page? (rank 2 → ½) | Ignores the others |
| **nDCG@10** | Are the right pages near the top, best first? Each lower rank counts less; 1 = perfect order | Least intuitive; the standard score in search papers |

**Caveats:** an unlabelled page that holds the answer counts as wrong; with 108 questions, differences under about
7 points may be noise.

**Learn more:** [Evidently AI: NDCG explained](https://www.evidentlyai.com/ranking-metrics/ndcg-metric) · deeper: [Pinecone: Evaluation Measures in IR](https://www.pinecone.io/learn/offline-evaluation/)

## 3. The search techniques

All of them answer one question, *how well does this page match this question?*, in different ways:

![Four ways to score a page: keywords, dense embeddings, late interaction, cross-encoder](figures/scoring_styles.png)

| technique | compares question and page by | used in practice | good at | weak at |
|---|---|---|---|---|
| **BM25** (keywords) | shared words, rare ones weighted more | default in Elasticsearch, OpenSearch, Lucene | exact names, numbers, codes; no model needed | synonyms; anything only in a chart |
| **Dense embeddings** | one vector each, cosine similarity | every vector database; Qwen3-Embedding ranks near the top of the MTEB leaderboard | meaning and paraphrases | one vector per page blurs details |
| **Late interaction** | many vectors each, best match per word (MaxSim) | ColBERT for text; ColPali / ColQwen for page images, the leading family on ViDoRe | fine detail; reads page images directly | stores hundreds of vectors per page |
| **Hybrid (RRF)** | merges the rankings of several methods | built into Elastic, OpenSearch, Weaviate, Vespa | robust: one method's misses are another's hits | only as good as its parts |
| **Reranker** (cross-encoder) | reads question and page together | Cohere Rerank, bge-reranker; the standard second stage | most accurate | slow, so only for the top 20 |

The usual production recipe: **hybrid retrieval (fast, wide) → reranker (slow, precise) → LLM.**
Not covered here: learned sparse search (SPLADE), query rewriting (HyDE), and agentic retrieval (Day 4).

## 4. Setup: data and scoring

In [57]:
# What this cell does:
# 1. find the repo folder and make a cache folder for page embeddings
# 2. load the 108 dev questions and the 2,313 pages (each page: image + text)
# 3. keep, for every question, its right pages and their grades (2 or 1)
import json
import math
import time
from pathlib import Path

import torch
from datasets import load_dataset

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "evals").is_dir())
CACHE = ROOT / "data" / "cache" / "search"          # page embeddings are saved here (git-ignored)
CACHE.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

dev = [json.loads(line) for line in (ROOT / "evals" / "dev.jsonl").read_text().splitlines()]
corpus = load_dataset("vidore/vidore_v3_pharmaceuticals", "corpus")["test"]
page_ids = corpus["corpus_id"]                      # position in the corpus -> page id
page_texts = corpus["markdown"]
questions = [q["query"] for q in dev]
grades = [{p["corpus_id"]: p["score"] for p in q["pages"]} for q in dev]   # right pages per question

print(len(questions), "questions,", len(page_ids), "pages, running on", device)

108 questions, 2313 pages, running on cuda


In [ ]:
# What this cell does:
# - the four scoring functions from section 2
# - evaluate(): scores one method on all questions and adds it to the leaderboard
# - show(): prints one question's top pages and marks the right ones
def hit_at_k(ranked, g, k):
    return float(any(c in g for c in ranked[:k]))

def recall_at_k(ranked, g, k):
    return len(set(ranked[:k]) & set(g)) / len(g)

def mrr_at_k(ranked, g, k):
    return next((1 / r for r, c in enumerate(ranked[:k], start=1) if c in g), 0.0)

def ndcg_at_k(ranked, g, k):
    dcg = sum(g.get(c, 0) / math.log2(i + 2) for i, c in enumerate(ranked[:k]))
    ideal = sum(x / math.log2(i + 2) for i, x in enumerate(sorted(g.values(), reverse=True)[:k]))
    return dcg / ideal

runs, leaderboard = {}, {}       # method -> ranked page ids per question;  method -> average scores

def evaluate(name, rankings):
    """Score one method: rankings = a list of page ids per question, best first."""
    runs[name] = rankings
    per_q = [{"hit@5": hit_at_k(r, g, 5), "recall@5": recall_at_k(r, g, 5),
              "mrr@10": mrr_at_k(r, g, 10), "ndcg@10": ndcg_at_k(r, g, 10)}
             for r, g in zip(rankings, grades, strict=True)]
    leaderboard[name] = {m: sum(x[m] for x in per_q) / len(per_q) for m in per_q[0]}
    print(f"{name:32}" + "  ".join(f"{m} {v:.3f}" for m, v in leaderboard[name].items()))

def show(name, i=0, k=5):
    """The top k pages one method found for question i, with the right ones marked."""
    print("Q:", questions[i])
    for rank, cid in enumerate(runs[name][i][:k], start=1):
        g = grades[i].get(cid)
        print(f"  {rank}. page {cid:5}  {'RIGHT, grade %d' % g if g else ''}")


## 5. BM25: matching words

**Learn more:** [Elastic: Practical BM25, part 2](https://www.elastic.co/blog/practical-bm25-part-2-the-bm25-algorithm-and-its-variables) · deeper: [IR book, section 11.4.3](https://nlp.stanford.edu/IR-book/)

# Baseline search: BM25 over page text
#### https://www.youtube.com/watch?v=TW9vHU1GpU4 quick watch on BM25 

#### Short Notes:
- Q1: How many times do these words appear in each document?
- Q2: Are they rare or common across the whole collection ?
- Q3: Are we looking at a short, focussed document or a huge blob of text? 

### BM25 score of page $D$ for question $Q$

![BM25 formula, annotated](BM25-figure.jpg)

| metric | question it answers |
|---|---|
| Hit@5 | is at least one correct page in the top 5? |
| Recall@5 | what share of all correct pages is in the top 5? |
| MRR@10 | how high is the first correct page? (1 = rank 1, 0.5 = rank 2, ...) |
| nDCG@10 | are the best pages (score 2) near the top? 1.0 = perfect order |

In [59]:
# What this cell does:
# 1. index: count every word on every page (stop words like "the" are dropped)
# 2. search: score all pages for every question and keep the top 100
# 3. score the rankings, then look at the first question
import bm25s

bm25 = bm25s.BM25()
bm25.index(bm25s.tokenize(page_texts, stopwords="english"))
pos, _ = bm25.retrieve(bm25s.tokenize(questions, stopwords="english"), k=100)

evaluate("BM25", [[page_ids[p] for p in row] for row in pos])
show("BM25")

BM25                            hit@5 0.833  recall@5 0.509  mrr@10 0.655  ndcg@10 0.564
Q: Does the Bolar exception help reduce the time it takes for generic drugs to reach the market after a patent expires?
  1. page   333  RIGHT, grade 2
  2. page   331  RIGHT, grade 2
  3. page   521  RIGHT, grade 1
  4. page   344  RIGHT, grade 2
  5. page   338  RIGHT, grade 1


## 6. Dense embeddings: matching meaning

A model turns any text into a **vector** (1,024 numbers) so that texts with similar meaning get nearby vectors.
Embed every page once; for each question, embed it and take the pages with the most similar vectors.
"side effects" can now match "adverse reactions", which BM25 would miss.

- Model: **Qwen3-Embedding-0.6B**. Questions get a one-line instruction in front. Use the model's own wording:
  a reworded instruction cost 0.07 nDCG.
- At millions of pages you would use an approximate index (FAISS, HNSW). For 2,313 pages, one matrix
  multiplication is exact and instant.

**Learn more:** [Pinecone: Dense Vectors](https://www.pinecone.io/learn/series/nlp/dense-vector-embeddings-nlp/) · deeper: [Sentence Transformers: Semantic Search](https://sbert.net/examples/sentence_transformer/applications/semantic-search/README.html), [Qwen3 Embedding paper](https://arxiv.org/abs/2506.05176)

In [60]:
# What this cell does:
# 1. load the embedding model
# 2. embed(): text -> one vector of 1,024 numbers (the last token's state sums up the text)
# 3. embed all pages once (saved to disk), then every question, with a one-line instruction in front
# 4. similarity = dot product of the normalised vectors; keep the 100 most similar pages
# 5. free the GPU memory
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

emb_name = "Qwen/Qwen3-Embedding-0.6B"
emb_tok = AutoTokenizer.from_pretrained(emb_name, padding_side="left")
emb_model = AutoModel.from_pretrained(emb_name, dtype=torch.float16).to(device).eval()

@torch.no_grad()
def embed(texts, max_length=1024, batch_size=16):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = emb_tok(texts[i:i + batch_size], padding=True, truncation=True, max_length=max_length,
                        return_tensors="pt").to(device)
        last = emb_model(**batch).last_hidden_state[:, -1]      # the last token summarises the text
        out.append(F.normalize(last.float(), dim=-1).cpu())
    return torch.cat(out)

cache = CACHE / "dense_pages.pt"
if cache.exists():
    page_vecs = torch.load(cache)
else:
    t = time.time()
    page_vecs = embed(page_texts)
    torch.save(page_vecs, cache)
    print(f"embedded {len(page_texts)} pages in {time.time() - t:.0f}s")

# the instruction the model was trained with; a reworded one scored 0.07 lower
task = "Given a web search query, retrieve relevant passages that answer the query"
query_vecs = embed([f"Instruct: {task}\nQuery:{q}" for q in questions], max_length=256)
print("page vectors:", tuple(page_vecs.shape))

sims = query_vecs @ page_vecs.T                                   # cosine similarity: questions x pages
evaluate("Dense (Qwen3-Embedding)", [[page_ids[p] for p in row] for row in sims.topk(100, dim=1).indices.tolist()])
show("Dense (Qwen3-Embedding)")

del emb_model
torch.cuda.empty_cache()

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 949.65it/s]


page vectors: (2313, 1024)
Dense (Qwen3-Embedding)         hit@5 0.852  recall@5 0.506  mrr@10 0.623  ndcg@10 0.545
Q: Does the Bolar exception help reduce the time it takes for generic drugs to reach the market after a patent expires?
  1. page   331  RIGHT, grade 2
  2. page   333  RIGHT, grade 2
  3. page   344  RIGHT, grade 2
  4. page   337  RIGHT, grade 2
  5. page   341  RIGHT, grade 1


## 7. Late interaction on page images: ColQwen2

Skip the text: look at the **page as a picture**. The model cuts each page image into small patches and makes
one short vector (128 numbers) per patch, a few hundred per page. The question gets one vector per word.

**MaxSim:** each question word finds its best-matching patch; the page score is the sum. A word like "dose"
can match the exact cell of a table, or a label on a chart, that text extraction may have mangled or lost.

- Model: **ColQwen2**. Cost: hundreds of vectors per page instead of one.

**Learn more:** [Jina AI: What is ColBERT and late interaction](https://jina.ai/news/what-is-colbert-and-late-interaction-and-why-they-matter-in-search/) · deeper: [Hugging Face: ColPali](https://huggingface.co/blog/manu/colpali), [ColPali paper](https://arxiv.org/abs/2407.01449)

In [61]:
# What this cell does:
# 1. load ColQwen2 and its processor (turns page images or questions into model inputs)
# 2. col_embed(): one vector of 128 numbers per image patch or per question token, padding removed
# 3. embed all page images once (saved to disk), then the questions
# 4. score_retrieval(): MaxSim for every question-page pair; keep the top 100
# 5. free the GPU memory
from transformers import ColQwen2ForRetrieval, ColQwen2Processor

col_name = "vidore/colqwen2-v1.0-hf"
col_proc = ColQwen2Processor.from_pretrained(col_name)
col_model = ColQwen2ForRetrieval.from_pretrained(col_name, dtype=torch.bfloat16).to(device).eval()

@torch.no_grad()
def col_embed(batch):
    """Multi-vector embeddings, one tensor per item, padding removed."""
    batch = batch.to(device)
    e = col_model(**batch).embeddings
    mask = batch["attention_mask"].bool()
    return [e[j][mask[j]].to(torch.float16).cpu() for j in range(len(e))]

cache = CACHE / "colqwen2_pages.pt"
if cache.exists():
    page_multi = torch.load(cache)
else:
    t, page_multi = time.time(), []
    for i in range(0, len(corpus), 4):
        images = [im.convert("RGB") for im in corpus[i:i + 4]["image"]]
        page_multi += col_embed(col_proc(images=images, return_tensors="pt"))
    torch.save(page_multi, cache)
    print(f"embedded {len(page_multi)} page images in {time.time() - t:.0f}s")

query_multi = []
for i in range(0, len(questions), 16):
    query_multi += col_embed(col_proc(text=questions[i:i + 16], return_tensors="pt"))
print("vectors per page: about", sum(len(p) for p in page_multi) // len(page_multi),
      "| per question: about", sum(len(q) for q in query_multi) // len(query_multi))

scores = col_proc.score_retrieval(query_multi, page_multi)       # MaxSim for every question-page pair
evaluate("ColQwen2 (page images)", [[page_ids[p] for p in row] for row in scores.topk(100, dim=1).indices.tolist()])
show("ColQwen2 (page images)")

del col_model
torch.cuda.empty_cache()

Loading weights: 100%|██████████| 731/731 [00:00<00:00, 16489.92it/s]


vectors per page: about 756 | per question: about 33
ColQwen2 (page images)          hit@5 0.870  recall@5 0.533  mrr@10 0.716  ndcg@10 0.615
Q: Does the Bolar exception help reduce the time it takes for generic drugs to reach the market after a patent expires?
  1. page   333  RIGHT, grade 2
  2. page   331  RIGHT, grade 2
  3. page   337  RIGHT, grade 2
  4. page   344  RIGHT, grade 2
  5. page   351  RIGHT, grade 2


## 8. Hybrid: merging rankings with RRF

Each method misses different questions. **Reciprocal Rank Fusion** merges their rankings without comparing
their raw scores (BM25 scores and cosine similarities are on different scales): every page gets
**Σ 1 / (60 + rank)** over the lists it appears in. A page ranked high by several methods wins.
The 60 is the standard constant from the original paper (Cormack et al., 2009).

**Learn more:** [Weaviate: Hybrid Search Explained](https://weaviate.io/blog/hybrid-search-explained) · deeper: [the RRF paper](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)

In [62]:
# What this cell does:
# 1. rrf(): for each question, every page gets 1/(60 + rank) from each list it appears in; sort by the sum
# 2. fuse the two text methods (BM25 + dense), then all three
def rrf(*methods, k=60, depth=100):
    """Fuse several runs: each run is a list of rankings, one per question."""
    fused = []
    for rankings in zip(*methods, strict=True):
        score = {}
        for ranked in rankings:
            for rank, cid in enumerate(ranked[:depth], start=1):
                score[cid] = score.get(cid, 0) + 1 / (k + rank)
        fused.append(sorted(score, key=score.get, reverse=True))
    return fused

evaluate("Hybrid text (BM25 + Dense)", rrf(runs["BM25"], runs["Dense (Qwen3-Embedding)"]))
evaluate("Hybrid all three", rrf(runs["BM25"], runs["Dense (Qwen3-Embedding)"], runs["ColQwen2 (page images)"]))

Hybrid text (BM25 + Dense)      hit@5 0.889  recall@5 0.558  mrr@10 0.687  ndcg@10 0.597
Hybrid all three                hit@5 0.889  recall@5 0.569  mrr@10 0.694  ndcg@10 0.618


## 9. Reranking: a careful second look at the top 20

A **cross-encoder** reads the question and one page **together** and outputs a relevance score. That is more
accurate than comparing vectors made separately, but it costs one model call per page, so it only reorders
the top 20 of the hybrid. Model: **bge-reranker-v2-m3** (reads page text). The ViDoRe V3 paper found text
rerankers add about 13 points of nDCG@10.

**Learn more:** [Pinecone: Rerankers and Two-Stage Retrieval](https://www.pinecone.io/learn/series/rag/rerankers/) · deeper: [Sentence Transformers: Cross-Encoders](https://sbert.net/examples/cross_encoder/applications/README.html)

In [63]:
# What this cell does:
# 1. load the reranker (a cross-encoder)
# 2. rerank(): score each (question, page text) pair together, then sort the pages by that score
# 3. rerank the hybrid's top 20 for every question; pages after the 20th keep their order
from transformers import AutoModelForSequenceClassification

rr_name = "BAAI/bge-reranker-v2-m3"
rr_tok = AutoTokenizer.from_pretrained(rr_name)
rr_model = AutoModelForSequenceClassification.from_pretrained(rr_name, dtype=torch.float16).to(device).eval()
text_of = dict(zip(page_ids, page_texts, strict=True))

@torch.no_grad()
def rerank(question, candidates, batch_size=10):
    scores = []
    for i in range(0, len(candidates), batch_size):
        pairs = [[question, text_of[c]] for c in candidates[i:i + batch_size]]
        batch = rr_tok(pairs, padding=True, truncation=True, max_length=1024, return_tensors="pt").to(device)
        scores += rr_model(**batch).logits.view(-1).float().tolist()
    order = sorted(range(len(candidates)), key=lambda j: -scores[j])
    return [candidates[j] for j in order]

t = time.time()
evaluate("Hybrid all three + rerank",
         [rerank(q, ranked[:20]) + ranked[20:] for q, ranked in zip(questions, runs["Hybrid all three"], strict=True)])
print(f"reranking took {time.time() - t:.0f}s for {len(questions)} questions")
show("Hybrid all three + rerank")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 670.82it/s]


Hybrid all three + rerank       hit@5 0.870  recall@5 0.593  mrr@10 0.705  ndcg@10 0.639
reranking took 8s for 108 questions
Q: Does the Bolar exception help reduce the time it takes for generic drugs to reach the market after a patent expires?
  1. page   344  RIGHT, grade 2
  2. page   333  RIGHT, grade 2
  3. page   331  RIGHT, grade 2
  4. page   337  RIGHT, grade 2
  5. page   341  RIGHT, grade 1


## 10. Experiments: change one thing, measure

### 10a. Why did dense lose to BM25? Try chunking

Three suspects: (1) the model squeezes a whole page into one vector, and long pages get cut off at 1,024 tokens;
(2) it is the small 0.6B model (the paper's 0.624 used the 8B); (3) pharma questions use exact terms (drug names,
codes) where keywords shine. Test suspect 1: split each page into overlapping ~200-word **chunks**, embed every
chunk, and score a page by its **best** chunk. Chunking is how most text RAG systems work.

**Learn more:** [Pinecone: Chunking Strategies](https://www.pinecone.io/learn/chunking-strategies/)

In [64]:
# What this cell does:
# 1. split every page into ~200-word chunks that overlap by 50 words; remember which page each came from
# 2. embed the chunks with the same model as section 6 (saved to disk)
# 3. page score = its best chunk's similarity; evaluate
emb_model = AutoModel.from_pretrained(emb_name, dtype=torch.float16).to(device).eval()

chunks, chunk_page = [], []
for i, text in enumerate(page_texts):
    words = text.split() or [""]
    for start in range(0, len(words), 150):
        chunks.append(" ".join(words[start:start + 200]))
        chunk_page.append(i)
        if start + 200 >= len(words):
            break
print(f"{len(page_texts)} pages -> {len(chunks)} chunks")

cache = CACHE / "dense_chunks_200w.pt"
if cache.exists():
    chunk_vecs = torch.load(cache)
else:
    t = time.time()
    chunk_vecs = embed(chunks, max_length=400)
    torch.save(chunk_vecs, cache)
    print(f"embedded {len(chunks)} chunks in {time.time() - t:.0f}s")

chunk_sims = query_vecs @ chunk_vecs.T                                    # questions x chunks
owner = torch.tensor(chunk_page).expand(len(questions), -1)                # which page each chunk is from
page_sims = torch.full((len(questions), len(page_texts)), -1.0).scatter_reduce(1, owner, chunk_sims, reduce="amax")
evaluate("Dense, chunked pages", [[page_ids[p] for p in row] for row in page_sims.topk(100, dim=1).indices.tolist()])

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1020.20it/s]


2313 pages -> 4258 chunks
Dense, chunked pages            hit@5 0.870  recall@5 0.530  mrr@10 0.634  ndcg@10 0.551


### 10b. Contextual retrieval: tell each page which document it is from

A page often does not say what it belongs to ("Table 3: results"). **Contextual retrieval** adds that context
before indexing. Anthropic's version writes a summary with an LLM; the cheap version here just puts the
document's name and type in front of every page. Re-run BM25 and dense on the new texts.

**Learn more:** [Anthropic: Introducing Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval)

In [65]:
# What this cell does:
# 1. look up every page's document: file name and type (slides, report, ...)
# 2. put it in front of the page text: "Document: <name> (<type>)"
# 3. re-run BM25 and dense on the new texts and evaluate both
docs = load_dataset("vidore/vidore_v3_pharmaceuticals", "documents_metadata")["test"]
title_of = {d["doc_id"]: f'{Path(d["file_name"]).stem.replace("_", " ")} ({d["doc_type"]})' for d in docs}
ctx_texts = [f"Document: {title_of[d]}\n\n{t}" for d, t in zip(corpus["doc_id"], page_texts, strict=True)]
print(ctx_texts[0][:160], "...\n")

bm25_ctx = bm25s.BM25()
bm25_ctx.index(bm25s.tokenize(ctx_texts, stopwords="english"))
pos, _ = bm25_ctx.retrieve(bm25s.tokenize(questions, stopwords="english"), k=100)
evaluate("BM25 + document title", [[page_ids[p] for p in row] for row in pos])

cache = CACHE / "dense_pages_ctx.pt"
if cache.exists():
    ctx_vecs = torch.load(cache)
else:
    ctx_vecs = embed(ctx_texts)
    torch.save(ctx_vecs, cache)
evaluate("Dense + document title",
         [[page_ids[p] for p in row] for row in (query_vecs @ ctx_vecs.T).topk(100, dim=1).indices.tolist()])

del emb_model
torch.cuda.empty_cache()

Document: final ce dental caries (slides)

U.S. FOOD& DRUG ADMINISTRATION
FDA
FDA Drug Topics:
An Update on Transmucosal
Buprenorphine and Dental Caries
Mark A. ...



BM25 + document title           hit@5 0.833  recall@5 0.526  mrr@10 0.673  ndcg@10 0.582
Dense + document title          hit@5 0.843  recall@5 0.518  mrr@10 0.646  ndcg@10 0.573


### 10c. Smaller vectors: Matryoshka and binary

A vector database pays for every number it stores. Qwen3-Embedding was trained so that its **first** 256 or 128
numbers still work on their own (*Matryoshka* embeddings). **Binary quantisation** keeps only the sign of each
number: 1 bit instead of 16. How much search quality does each cost?

**Learn more:** [Hugging Face: Matryoshka Embeddings](https://huggingface.co/blog/matryoshka) · deeper: [Hugging Face: Embedding Quantization](https://huggingface.co/blog/embedding-quantization)

In [66]:
# What this cell does:
# 1. Matryoshka: keep the first N of the 1,024 numbers, re-normalise, search again
# 2. binary: keep only the sign (+1 or -1) of every number, search again
# 3. print how much memory one page's vector takes in each case
def topk_ids(q, p):
    return [[page_ids[i] for i in row] for row in (q @ p.T).topk(100, dim=1).indices.tolist()]

for dims in [256, 128]:
    evaluate(f"Dense, first {dims} numbers",
             topk_ids(F.normalize(query_vecs[:, :dims], dim=-1), F.normalize(page_vecs[:, :dims], dim=-1)))
evaluate("Dense, binary (1 bit each)", topk_ids(torch.sign(query_vecs), torch.sign(page_vecs)))
print("\nbytes per page: 1,024 numbers at 16 bits = 2,048 | 256 numbers = 512 | 128 numbers = 256 | binary = 128")

Dense, first 256 numbers        hit@5 0.880  recall@5 0.510  mrr@10 0.627  ndcg@10 0.531
Dense, first 128 numbers        hit@5 0.815  recall@5 0.468  mrr@10 0.609  ndcg@10 0.489
Dense, binary (1 bit each)      hit@5 0.815  recall@5 0.469  mrr@10 0.570  ndcg@10 0.497

bytes per page: 1,024 numbers at 16 bits = 2,048 | 256 numbers = 512 | 128 numbers = 256 | binary = 128


### 10d. Vector indexes: FAISS

So far every question was compared with every page: **exact** search. At millions of pages that is too slow,
so vector databases use an **index** that finds *nearly* the closest pages much faster:

- **Flat:** exact, compares with every page (what we did so far).
- **HNSW:** a layered graph of neighbours; the search hops from far to near. Knob: `efSearch`, how many candidates it explores.
- **IVF:** groups pages into clusters; searches only the few clusters nearest the question. Knob: `nprobe`, how many clusters.

We check two things: does the index find the **same top 10** as exact search, and how much **faster** is it?

**Learn more:** [Pinecone: HNSW explained](https://www.pinecone.io/learn/series/faiss/hnsw/) · deeper: [Pinecone: Nearest Neighbor Indexes](https://www.pinecone.io/learn/series/faiss/vector-indexes/), [FAISS wiki](https://github.com/facebookresearch/faiss/wiki)

In [67]:
# What this cell does:
# 1. put the 2,313 page vectors into three FAISS indexes: Flat (exact), HNSW and IVF
# 2. search every question with each; score them, time them, and compare with the exact top 10
import faiss
import numpy as np

pv = page_vecs.numpy().astype("float32")        # vectors are normalised, so inner product = cosine
qv = query_vecs.numpy().astype("float32")
dim = pv.shape[1]

flat = faiss.IndexFlatIP(dim)
hnsw = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)   # 32 neighbours per node
hnsw.hnsw.efSearch = 64
ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(dim), dim, 48, faiss.METRIC_INNER_PRODUCT)   # 48 clusters
ivf.train(pv)
ivf.nprobe = 4                                                     # search the 4 nearest clusters
for index in (flat, hnsw, ivf):
    index.add(pv)

def timed_search(index, queries, k):
    t = time.perf_counter()
    _, found = index.search(queries, k)
    return found, (time.perf_counter() - t) * 1000 / len(queries)

exact, _ = timed_search(flat, qv, 10)
for name, index in [("FAISS Flat (exact)", flat), ("FAISS HNSW", hnsw), ("FAISS IVF", ivf)]:
    found, ms = timed_search(index, qv, 100)
    same = np.mean([len(set(f[:10]) & set(e)) / 10 for f, e in zip(found, exact, strict=True)])
    evaluate(name, [[page_ids[p] for p in row if p >= 0] for row in found])
    print(f"    {ms:.3f} ms per question | finds {same:.0%} of the exact top 10")

FAISS Flat (exact)              hit@5 0.852  recall@5 0.506  mrr@10 0.623  ndcg@10 0.545
    0.149 ms per question | finds 100% of the exact top 10
FAISS HNSW                      hit@5 0.852  recall@5 0.506  mrr@10 0.623  ndcg@10 0.545
    0.093 ms per question | finds 100% of the exact top 10
FAISS IVF                       hit@5 0.806  recall@5 0.482  mrr@10 0.572  ndcg@10 0.504
    0.034 ms per question | finds 87% of the exact top 10


At 2,313 pages every index is instant, so the trade-off is invisible. Make the collection 40 times bigger
(each page copied 40 times with a little noise, about 92,000 vectors) and time them again:

In [68]:
# What this cell does:
# 1. build a 40x bigger collection: every page vector plus small random noise
# 2. time exact search vs HNSW vs IVF on it, and check how much of the exact top 10 each finds
rng = np.random.default_rng(0)
big = np.concatenate([pv + 0.02 * rng.standard_normal(pv.shape).astype("float32") for _ in range(40)])
faiss.normalize_L2(big)

big_flat = faiss.IndexFlatIP(dim)
big_hnsw = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
big_hnsw.hnsw.efSearch = 64
big_ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(dim), dim, 1000, faiss.METRIC_INNER_PRODUCT)
big_ivf.train(big)
big_ivf.nprobe = 10
for name, index in [("Flat", big_flat), ("HNSW", big_hnsw), ("IVF", big_ivf)]:
    t = time.time()
    index.add(big)
    print(f"{name:5} built in {time.time() - t:5.1f}s")

big_exact, flat_ms = timed_search(big_flat, qv, 10)
print(f"\n{len(big):,} vectors | exact search: {flat_ms:.2f} ms per question")
for name, index in [("HNSW", big_hnsw), ("IVF", big_ivf)]:
    found, ms = timed_search(index, qv, 10)
    same = np.mean([len(set(f) & set(e)) / 10 for f, e in zip(found, big_exact, strict=True)])
    print(f"{name:5} {ms:.2f} ms per question ({flat_ms / ms:.0f}x faster) | finds {same:.0%} of the exact top 10")

Flat  built in   0.1s
HNSW  built in   9.6s
IVF   built in   1.8s

92,520 vectors | exact search: 8.95 ms per question
HNSW  0.11 ms per question (84x faster) | finds 90% of the exact top 10
IVF   0.27 ms per question (33x faster) | finds 99% of the exact top 10


### Not run here, and why

- **Query rewriting, HyDE:** need the answer model to write the new query; they come with the agent (Day 4).
- **Learned sparse (SPLADE):** one more model for a similar idea to 10b; optional.

## 11. Side by side

nDCG@10 overall, on text-only questions, and on questions whose evidence includes a chart, table, infographic or
image. The paper column is ViDoRe V3's number for the same dataset (all English questions, not just dev).

In [69]:
# What this cell does:
# 1. mark the questions whose evidence includes a chart, table, infographic or image
# 2. for every method: nDCG@10 overall, on text-only questions, on visual questions, and Recall@5
visual = [bool(set(q["content_type"]) - {"Text"}) for q in dev]
paper = {"BM25": "0.569", "Dense (Qwen3-Embedding)": "0.624 (8B model)", "ColQwen2 (page images)": "0.609 (ColQwen2.5)"}

def mean_ndcg(name, keep):
    vals = [ndcg_at_k(r, g, 10) for r, g, kp in zip(runs[name], grades, keep, strict=True) if kp]
    return sum(vals) / len(vals)

print(f"{'method':32}{'nDCG@10':>9}{'text':>8}{'visual':>8}{'Recall@5':>10}   paper")
for name, s in sorted(leaderboard.items(), key=lambda kv: -kv[1]["ndcg@10"]):
    print(f"{name:32}{s['ndcg@10']:9.3f}{mean_ndcg(name, [not v for v in visual]):8.3f}"
          f"{mean_ndcg(name, visual):8.3f}{s['recall@5']:10.3f}   {paper.get(name, '')}")
print(f"\n{sum(visual)} visual and {len(visual) - sum(visual)} text-only questions. "
      "With this few, differences under about 0.07 may be noise.")

method                            nDCG@10    text  visual  Recall@5   paper
Hybrid all three + rerank           0.639   0.714   0.582     0.593   
Hybrid all three                    0.618   0.683   0.567     0.569   
ColQwen2 (page images)              0.615   0.657   0.583     0.533   0.609 (ColQwen2.5)
Hybrid text (BM25 + Dense)          0.597   0.666   0.544     0.558   
BM25 + document title               0.582   0.599   0.568     0.526   
Dense + document title              0.573   0.610   0.544     0.518   
BM25                                0.564   0.594   0.540     0.509   0.569
Dense, chunked pages                0.551   0.613   0.503     0.530   
Dense (Qwen3-Embedding)             0.545   0.596   0.505     0.506   0.624 (8B model)
FAISS Flat (exact)                  0.545   0.596   0.505     0.506   
FAISS HNSW                          0.545   0.596   0.505     0.506   
Dense, first 256 numbers            0.531   0.579   0.495     0.510   
FAISS IVF                        

In [70]:
# What this cell does:
# 1. save every method's rankings (the answering step reads them)
# 2. log every method's scores to reports/runs/
# keep every method's rankings for the answering step, and log the scores
from vdoc_agent.runlog import log_run

(CACHE / "runs.json").write_text(json.dumps(runs))
for name, s in leaderboard.items():
    log_run(f"search-{name}", {"method": name, "split": "dev"}, s)
print("saved", CACHE / "runs.json")

saved /storage/homefs/kr23w045/vdoc-agent/data/cache/search/runs.json


## What to look for

- Which method wins overall, and is the gap bigger than the noise (about 0.07)?
- Does page-image search (ColQwen2) win on the **visual** questions and lose on text ones?
- Did chunking (10a) close the gap between dense and BM25? If not, suspects 2 and 3 remain.
- Did the document title (10b) help BM25, dense, or neither?
- How much nDCG does each smaller vector (10c) cost, for how much memory saved?
- How much of the exact top 10 do HNSW and IVF find (10d), and how much faster are they at 92,000 vectors?
- Does the hybrid beat every single method? Does reranking add anything like the paper's +13 points?